# 1002 · 00 · Set up the environment

Lab 1002 · From bronze to gold (Ventolia Energy).

Run it once (**Run all**). It creates the `lab1002` catalog with one schema per
medallion layer and copies the three raw files into the `landing.data` volume:

| File | What it is |
|---|---|
| `turbines.csv` | Master data: the 36 turbines, their wind farm and rated power |
| `scada_2026-10-21.csv` | One reading every 10 minutes per turbine: wind, power, status |
| `work_orders.csv` | Maintenance work orders of the day |

In [ ]:
%python
import os
import shutil

CATALOG = "lab1002"

# This notebook's folder is the lab folder inside your Git folder.
LAB_PATH = os.getcwd()
if not LAB_PATH.startswith("/Workspace"):
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    LAB_PATH = "/Workspace" + os.path.dirname(ctx.notebookPath().get())

print("Lab:    ", LAB_PATH)
print("Catalog:", CATALOG)

## 1 · Catalog, one schema per layer, and the landing volume

In [ ]:
%python
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
for schema in ["landing", "bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.landing.data")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

## 2 · Raw files into the volume

You should see **three files**.

In [ ]:
%python
source = f"{LAB_PATH}/data"
target = f"/Volumes/{CATALOG}/landing/data"

for file_name in sorted(os.listdir(source)):
    shutil.copyfile(f"{source}/{file_name}", f"{target}/{file_name}")
    print("copied", file_name)

display(dbutils.fs.ls(target))